# MOSS-TTSD v1.0 — Two-Speaker Travel Dialogue (Colab GPU)

Runs **MOSS-TTSD v1.0** on Colab CUDA. Do **not** expect this on an M4 MacBook Air 16GB — the 8B dialogue model + tokenizer belong on GPU cloud.

**Local (Mac) pipeline stays light:**
1. `python run_moss_director.py chokhi_dhani.txt -o output/chokhi_dhani_moss.txt`  
   → travel editor + MOSS director + pronunciations → `.txt` + `.cues.json`
2. Upload the `.txt` + `male_ref.wav` / `female_ref.wav` (+ optional `.cues.json`) here
3. Generate `episode.wav` on GPU
4. Back on Mac: `python run_mix_moss.py --episode episode.wav --cues output/chokhi_dhani_moss.cues.json`

**GPU:** `Runtime → Change runtime type → GPU`. v1.0 is too big for full-GPU on T4 — generate cell uses **GPU + CPU RAM offload** (`max_memory` cpu≤10GiB). L4/A100 is still nicer. Refuses pure CPU.

**Languages:** v1.0 includes English + Spanish (and more). Hindi is not in the official set.


In [ ]:
# 1. GPU check + hybrid memory budget (GPU + CPU RAM ≤10GiB)
import gc
import os
import subprocess

import torch

assert torch.cuda.is_available() and torch.cuda.device_count() >= 1, (
    "No CUDA GPU. Runtime → Change runtime type → GPU, then re-run."
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Keep CPU offload budget at 10GiB as requested
CPU_MAX_GIB = 10
# Leave headroom on T4 (~14.5GiB) for activations / codec
GPU_MAX_GIB = 11

gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
BF16_OK = capability >= (8, 0)
DTYPE = "bf16" if BF16_OK else "fp16"
ATTN = "sdpa"  # never flash-attn on Colab free

gc.collect()
torch.cuda.empty_cache()

try:
    print("nvidia-smi:", subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
        text=True,
    ).strip())
except Exception as e:
    print("nvidia-smi:", e)

print(f"GPU: {gpu_name} | VRAM {vram_gb:.1f} GB | dtype={DTYPE} | attn={ATTN}")
print(f"Offload budget: GPU≤{GPU_MAX_GIB}GiB + CPU≤{CPU_MAX_GIB}GiB (not full .to(cuda))")
if vram_gb < 16:
    print("T4-class GPU: use chunking (TURNS_PER_CHUNK=6) if generate still OOMs.")


In [ ]:
# 2. Upload inputs from local MOSS director
# Required: directed .txt (from run_moss_director.py)
# Required for cloning: male_ref.wav + female_ref.wav
# Optional: *.cues.json (for local mix later — download stays with you; upload optional)
from pathlib import Path

from google.colab import files

INPUT_DIR = Path("/content/inputs")
OUT_DIR = Path("/content/outputs")
INPUT_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

existing = sorted(p.name for p in INPUT_DIR.iterdir())
if existing:
    print(f"Already in {INPUT_DIR}: {existing}")
    print("Delete files there and re-run this cell to replace them.")
else:
    print("Upload: chokhi_dhani_moss.txt + male_ref.wav + female_ref.wav (+ optional .cues.json)")
    uploaded = files.upload()
    for name, data in uploaded.items():
        (INPUT_DIR / name).write_bytes(data)
    print("Saved:", sorted(uploaded))

scripts = sorted(INPUT_DIR.glob("*.txt"))
assert scripts, "No .txt script — run run_moss_director.py locally first, then upload"
SCRIPT_FILE = scripts[0]
have_refs = (INPUT_DIR / "male_ref.wav").exists() and (INPUT_DIR / "female_ref.wav").exists()
CUES_FILE = next(INPUT_DIR.glob("*.cues.json"), None)
print(f"Script: {SCRIPT_FILE.name}")
print(f"Refs: {'OK' if have_refs else 'MISSING — cell 3 can synthesize OR re-upload'}")
print(f"Cues: {CUES_FILE.name if CUES_FILE else 'none (mix holds/beds need local cues.json)'}")


In [ ]:
# 3. Optional: synthesize refs with Qwen3-TTS if not uploaded
# Runs BEFORE MOSS install (different transformers pin). Skip if refs already present.
GENERATE_REFS = not have_refs
N_TAKES = 4  # keep small on Colab; local make_voices_refs.py is better for quality

REF_SPECS = {
    "male": {
        "filename": "male_ref.wav",
        "text": (
            "Oh wow, okay — you have got to see this place, the colours are just "
            "unbelievable! I mean… I have been to so many markets before, but this? "
            "Hah, this is something else entirely. Honestly, we could spend the whole "
            "day right here, just wandering around slowly, and I would not complain one bit."
        ),
        "instruct": (
            "Male, late 20s, warm Indian accent, natural conversational storyteller, "
            "energy that shifts naturally between excited bursts and calm reflective "
            "moments, expressive rises and falls in intonation, occasional soft natural "
            "laugh, sounds like a real person talking to a friend, not performing, not flat"
        ),
    },
    "female": {
        "filename": "female_ref.wav",
        "text": (
            "You know what I love about places like this? Every corner has a little "
            "story hiding in it. Hah — like, who decided to paint that door turquoise? "
            "I could genuinely wander around here for hours… it just feels alive, "
            "doesn't it? Okay come on, let's go find out where that music is coming from!"
        ),
        "instruct": (
            "Female, early 30s, warm Indian accent, lively storytelling friend, energy "
            "that moves naturally between curious wonder, playful amusement and warm calm, "
            "expressive intonation with gentle rises and falls, occasional soft laugh, "
            "sounds like a real person chatting, not performing, not flat"
        ),
    },
}

if GENERATE_REFS:
    !pip install -q qwen-tts librosa soundfile
    import gc, json, shutil
    import librosa, numpy as np, soundfile as sf, torch
    from qwen_tts import Qwen3TTSModel

    qwen = Qwen3TTSModel.from_pretrained(
        "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
        device_map="cuda",
        dtype=torch.bfloat16 if BF16_OK else torch.float16,
        attn_implementation="sdpa",
    )
    CAND_DIR = INPUT_DIR / "ref_candidates"
    CAND_DIR.mkdir(exist_ok=True)

    def dynamics_score(wav, sr):
        f0, voiced, _ = librosa.pyin(wav, fmin=60, fmax=400, sr=sr)
        f0 = f0[np.asarray(voiced, dtype=bool)]
        f0 = f0[~np.isnan(f0)]
        pitch_var = float(np.std(np.log(f0))) if len(f0) > 10 else 0.0
        rms = librosa.feature.rms(y=wav)[0]
        energy_var = float(np.std(rms) / (np.mean(rms) + 1e-8))
        return pitch_var + energy_var

    ranked = {}
    ref_texts = {}
    for key, spec in REF_SPECS.items():
        print(f"=== {key}: {N_TAKES} takes ===")
        scored = []
        for i in range(N_TAKES):
            torch.manual_seed(1000 + i)
            wavs, sr = qwen.generate_voice_design(
                text=spec["text"], language="English", instruct=spec["instruct"]
            )
            w = wavs[0]
            wav = w.detach().cpu().numpy() if hasattr(w, "detach") else np.asarray(w)
            wav = np.asarray(wav, dtype=np.float32).squeeze()
            path = CAND_DIR / f"{key}_take{i:02d}.wav"
            sf.write(path, wav, sr)
            score = dynamics_score(wav, sr)
            scored.append((score, path))
            print(f"  take {i:02d}: dynamics={score:.3f}")
        scored.sort(key=lambda x: x[0], reverse=True)
        ranked[key] = scored
        shutil.copy(scored[0][1], INPUT_DIR / spec["filename"])
        ref_texts[key] = spec["text"]
    (INPUT_DIR / "ref_texts.json").write_text(json.dumps(ref_texts, indent=2))
    del qwen
    gc.collect()
    torch.cuda.empty_cache()
    print("Refs saved; Qwen unloaded for MOSS v1.0.")
else:
    print("Using uploaded refs — skipping Qwen synthesis.")


## Check refs
Listen before spending 15–30+ min on v1.0 generate. Refs set **timbre / accent / baseline energy**.


In [ ]:
# 4. Listen to refs
import IPython.display as ipd
for name in ("male_ref.wav", "female_ref.wav"):
    p = INPUT_DIR / name
    assert p.exists(), p
    print(name)
    display(ipd.Audio(str(p)))


In [ ]:
# 5. Install MOSS-TTSD v1.0 + accelerate (needed for CPU/GPU offload)
%cd /content
![ -d MOSS-TTSD ] || git clone --depth 1 https://github.com/OpenMOSS/MOSS-TTSD.git
%cd /content/MOSS-TTSD

!pip install -q -r requirements.txt
!pip install -q soundfile torchaudio huggingface_hub accelerate
print("Installed. Repo:", __import__("pathlib").Path.cwd())


In [ ]:
# 6. Prefetch weights (optional but avoids surprise mid-run downloads)
from huggingface_hub import snapshot_download

MODEL_ID = "OpenMOSS-Team/MOSS-TTSD-v1.0"
CODEC_ID = "OpenMOSS-Team/MOSS-Audio-Tokenizer"

print("Downloading", MODEL_ID, "…")
snapshot_download(MODEL_ID)
print("Downloading", CODEC_ID, "…")
snapshot_download(CODEC_ID)
print("Weights cached.")


In [ ]:
# 7. Config — refs + chunking + token budget
import json
from pathlib import Path

INPUT_DIR = Path("/content/inputs")
OUT_DIR = Path("/content/outputs")
SPEAKER1_REF = INPUT_DIR / "male_ref.wav"
SPEAKER2_REF = INPUT_DIR / "female_ref.wav"
SCRIPT_FILE = sorted(INPUT_DIR.glob("*.txt"))[0]

ref_texts_path = INPUT_DIR / "ref_texts.json"
if ref_texts_path.exists():
    _texts = json.loads(ref_texts_path.read_text())
    REF_TEXT_M, REF_TEXT_F = _texts["male"], _texts["female"]
    print("Ref transcripts from ref_texts.json")
else:
    # Defaults match make_voices_refs.py / hosts — EDIT if your uploaded refs differ
    REF_TEXT_M = REF_TEXT_F = (
        "Whoa, this place is incredible! Can you believe how vibrant everything feels? "
        "I am loving this already."
    )
    print("Using default ref transcripts — edit if your wav words differ.")

# 0 = one pass (best continuity). On T4 prefer 6 if generate OOMs mid-sample.
TURNS_PER_CHUNK = 6
SEED = 42

assert SPEAKER1_REF.exists() and SPEAKER2_REF.exists()
print(f"Script: {SCRIPT_FILE.name}")
print(f"S1={SPEAKER1_REF.name} | S2={SPEAKER2_REF.name} | TURNS_PER_CHUNK={TURNS_PER_CHUNK}")


In [ ]:
# 8. Parse directed script → JSONL for v1.0 inference.py
import json
import re
from pathlib import Path

SPEAKER_LINE = re.compile(r"^\s*Speaker\s+(?P<num>1|2)\s*:\s*(?P<text>.+?)\s*$", re.IGNORECASE)
TAG_LINE = re.compile(r"^\s*\[(?P<tag>M|F)\]\s*(?P<text>.+?)\s*$", re.IGNORECASE)
TAG_TO_NUM = {"M": "1", "F": "2"}

def parse_turns(text: str):
    turns = []
    for line in text.splitlines():
        if not line.strip():
            continue
        if m := SPEAKER_LINE.match(line):
            turns.append((m.group("num"), m.group("text").strip()))
        elif m := TAG_LINE.match(line):
            turns.append((TAG_TO_NUM[m.group("tag").upper()], m.group("text").strip()))
    return turns

def turns_to_ttsd_text(turns):
    # v1.0 expects [S1]/[S2] in the text field
    return "".join(f"[S{num}]{text}" for num, text in turns)

turns = parse_turns(SCRIPT_FILE.read_text(encoding="utf-8"))
assert turns, "No Speaker 1/2 or [M]/[F] lines found"

if TURNS_PER_CHUNK and TURNS_PER_CHUNK > 0:
    chunks = [turns[i : i + TURNS_PER_CHUNK] for i in range(0, len(turns), TURNS_PER_CHUNK)]
else:
    chunks = [turns]

base_entry = {
    "base_path": str(INPUT_DIR),
    "prompt_audio_speaker1": SPEAKER1_REF.name,
    "prompt_text_speaker1": REF_TEXT_M,
    "prompt_audio_speaker2": SPEAKER2_REF.name,
    "prompt_text_speaker2": REF_TEXT_F,
}

JSONL_PATH = INPUT_DIR / "input.jsonl"
with JSONL_PATH.open("w", encoding="utf-8") as f:
    for chunk in chunks:
        f.write(json.dumps({**base_entry, "text": turns_to_ttsd_text(chunk)}, ensure_ascii=False) + "\n")

total_words = sum(len(t.split()) for _, t in turns)
est_minutes = total_words / 160.0
# README: ~12.5 tokens per second of audio
est_tokens = int(est_minutes * 60 * 12.5) + 400
MAX_NEW_TOKENS = max(1500, min(8000, est_tokens))

print(f"{len(turns)} turns, {total_words} words → ~{est_minutes:.1f} min")
print(f"{len(chunks)} generation pass(es); max_new_tokens≈{MAX_NEW_TOKENS}")
print(f"Wrote {JSONL_PATH}")
print("Preview:", turns_to_ttsd_text(turns)[:220], "...")


In [ ]:
# 9. Generate — GPU + CPU offload (RAM ≤10GiB). Do NOT use stock inference.py .to(cuda).
# Stock path OOMs on T4 (~14GB) because the full 8B model + codec won't fit.
%cd /content/MOSS-TTSD

import gc
import json
import sys
from pathlib import Path

import soundfile as sf
import torch
from transformers import AutoModel, AutoProcessor

from generation_utils import prepare_sample

assert torch.cuda.is_available(), "GPU lost — reconnect as GPU runtime"

# Free leftovers from Qwen ref synthesis / prior failed loads
gc.collect()
torch.cuda.empty_cache()
print("VRAM before load:", round(torch.cuda.memory_allocated()/1e9, 2), "GB allocated")

MODEL_ID = "OpenMOSS-Team/MOSS-TTSD-v1.0"
CODEC_ID = "OpenMOSS-Team/MOSS-Audio-Tokenizer"
OUT_DIR = Path("/content/outputs")
OUT_DIR.mkdir(exist_ok=True)
JSONL_PATH = Path("/content/inputs/input.jsonl")

dtype = torch.bfloat16 if BF16_OK else torch.float16
max_memory = {
    0: f"{GPU_MAX_GIB}GiB",
    "cpu": f"{CPU_MAX_GIB}GiB",
}

print("Loading processor + audio tokenizer on CPU (saves GPU for LM)...")
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    codec_path=CODEC_ID,
)
# Keep codec on CPU — encode refs on CPU, generate with offloaded LM
if getattr(processor, "audio_tokenizer", None) is not None:
    processor.audio_tokenizer = processor.audio_tokenizer.to("cpu")
    processor.audio_tokenizer.eval()

print(f"Loading MOSS-TTSD with device_map=auto, max_memory={max_memory}, attn=sdpa...")
model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    attn_implementation="sdpa",
    torch_dtype=dtype,
    device_map="auto",
    max_memory=max_memory,
    low_cpu_mem_usage=True,
)
model.eval()
print("VRAM after load:", round(torch.cuda.memory_allocated()/1e9, 2), "GB allocated")
print("device_map:", getattr(model, "hf_device_map", None))

target_sr = int(processor.model_config.sampling_rate)
samples = []
with JSONL_PATH.open(encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        samples.append((line_no, json.loads(line)))

print(f"{len(samples)} JSONL sample(s); max_new_tokens={MAX_NEW_TOKENS}")

with torch.inference_mode():
    for line_no, raw in samples:
        sample_id, _record, conversation = prepare_sample(
            line_no=line_no,
            raw_sample=raw,
            mode="voice_clone_and_continuation",
            processor=processor,
            target_sr=target_sr,
            text_normalize_enabled=True,
            sample_rate_normalize_enabled=True,
        )
        batch = processor([conversation], mode="continuation")
        # Inputs on first model device (GPU shard 0)
        first_device = torch.device("cuda:0")
        input_ids = batch["input_ids"].to(first_device)
        attention_mask = batch["attention_mask"].to(first_device)

        print(f"Generating sample {sample_id}…")
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=int(MAX_NEW_TOKENS),
            audio_temperature=1.1,
            audio_top_p=0.9,
            audio_top_k=50,
            audio_repetition_penalty=1.1,
        )
        messages = processor.decode(outputs)
        message = messages[0]
        wav_segments = []
        if message is not None:
            for wav in message.audio_codes_list:
                if isinstance(wav, torch.Tensor):
                    wav_segments.append(
                        wav.detach().to(dtype=torch.float32, device="cpu").reshape(-1)
                    )
        assert wav_segments, f"No audio for sample {sample_id}"
        merged = torch.cat(wav_segments, dim=0)
        out_path = OUT_DIR / f"{sample_id}.wav"
        sf.write(str(out_path), merged.numpy(), target_sr)
        print(f"  -> {out_path} ({merged.shape[-1] / target_sr:.1f}s)")
        del outputs, messages, input_ids, attention_mask, batch
        gc.collect()
        torch.cuda.empty_cache()

print("Generation finished ->", OUT_DIR)


In [ ]:
# 10. Assemble episode + listen
# v1.0 inference writes wavs under save_dir (names vary); collect all *.wav except tiny junk.
from pathlib import Path
import IPython.display as ipd
from pydub import AudioSegment

OUT_DIR = Path("/content/outputs")
wavs = sorted(OUT_DIR.rglob("*.wav"))
# Prefer numbered outputs from batch runner
preferred = sorted(OUT_DIR.glob("*.wav")) + sorted(OUT_DIR.glob("*/*.wav"))
clips = []
seen = set()
for p in preferred + wavs:
    rp = str(p.resolve())
    if rp in seen or p.stat().st_size < 1000:
        continue
    seen.add(rp)
    clips.append(p)

assert clips, f"No wav outputs in {OUT_DIR} — scroll generate cell logs for errors"
print("Found clips:", [c.name for c in clips])

episode = AudioSegment.empty()
for clip in clips:
    episode += AudioSegment.from_wav(clip)

EPISODE_PATH = OUT_DIR / "episode.wav"
episode.export(EPISODE_PATH, format="wav")
print(f"{len(clips)} clip(s) → {EPISODE_PATH} ({len(episode)/60000:.1f} min)")
ipd.Audio(str(EPISODE_PATH))


In [ ]:
# 11. Download — then mix on Mac (holds + beds)
from google.colab import files

files.download(str(EPISODE_PATH))
for name in ("male_ref.wav", "female_ref.wav", "ref_texts.json"):
    p = INPUT_DIR / name
    if p.exists():
        files.download(str(p))

print(
    "On your Mac:\\n"
    "  python run_mix_moss.py --episode ~/Downloads/episode.wav "
    "--cues output/chokhi_dhani_moss.cues.json -o output/episode_mixed.mp3"
)


## Troubleshooting (v1.0 + this repo)

- **CUDA OOM on load (stock inference.py)** — expected on T4 if you `.to(cuda)` the full model. Use **cell 9** (device_map + CPU≤10GiB offload). Do not call stock `inference.py` on T4.
- **Still OOM during generate** — set `TURNS_PER_CHUNK = 6` in config, re-run parse + generate; or use Colab Pro L4/A100.
- **FlashAttention2 ImportError** — ignore; we force `attn_implementation="sdpa"`.
- **HF rate limit / slow download** — optional: `os.environ["HF_TOKEN"] = "hf_..."` before prefetch.
- **Mac** — director + mix only; never load v1.0 locally on 16GB.
- **Flat / robotic F** — better F ref + re-run `run_moss_director.py`.
- **Pronunciation** — `config/pronunciations.yaml` → re-director → re-upload `.txt`.
- **Holds / ambience** — `run_mix_moss.py` on Mac with `.cues.json`.
- **After restart** — re-run GPU check → install (if needed) → config → parse → generate.
